# Chapter-9：多智能体失败归因与评估（AutoFA）

本 notebook 演示如何将 **OpenTelemetry span JSONL**（如 `latc.travel-multi-agent` 导出的轨迹）转换为 **Who&When 格式**，再使用 `Automated_FA` 进行失败归因，并与人工标注对比准确率。

详细命令行说明见 [`testdata/README.md`](testdata/README.md)。

**核心学习目标**

| 步骤 | 学什么 | 对应代码 |
|------|--------|----------|
| **Step 1** | span JSONL → Who&When JSON | `convert_spans_to_whowhen.py` |
| **Step 2** | 对 history 执行失败归因 | `Automated_FA/inference.py` |
| **Step 3** | 查看 Agent / Step / Reason | `outputs/*.txt` |
| **Step 4（可选）** | 与人工标注对比准确率 | `Automated_FA/evaluate.py` |

主函数 `run_failure_attribution_eval` 把上述步骤串成一条链路（结构类似记忆章节的 `chat_with_memories`：**准备输入 → 调用模型 → 评估结果**）。

## 0. 环境与前置条件

```bash
pip install openai python-dotenv tqdm
```

在 `Automated_FA/.env` 中填写（参考 `.env.example`）：

```env
DASHSCOPE_API_KEY=sk-你的DashScope密钥
DASHSCOPE_BASE_URL=https://dashscope.aliyuncs.com/compatible-mode/v1
DASHSCOPE_MODEL=qwen-plus
```

Key 获取：[DashScope 控制台](https://dashscope.console.aliyun.com/)

**轨迹文件要求（JSONL，每行一个 span）：**

转换脚本按后缀匹配 span 名（不写死完整前缀）：

- `*.planner.*`（排除 `.build_plan` 包装 span）
- `*.agent.invoke`
- `*.orchestration.aggregate`

> **Step 2 归因** 需要 API Key；**Step 1 转换** 和 **Step 4 评估**（复用已有 `outputs/*.txt`）不需要 Key。

In [76]:
import json
import os
import subprocess
import sys
from pathlib import Path
from typing import Any

from dotenv import load_dotenv

if sys.platform == "win32":
    for _s in (sys.stdout, sys.stderr):
        try:
            _s.reconfigure(encoding="utf-8")
        except Exception:
            pass

def _resolve_chapter_root() -> Path:
    """兼容从 Chapter-9/ 或项目根目录打开 notebook。"""
    for base in (Path.cwd(), *Path.cwd().parents):
        if (base / "Automated_FA").exists() and (base / "testdata" / "convert_spans_to_whowhen.py").exists():
            return base
        chapter9 = base / "Chapter-9"
        if (chapter9 / "Automated_FA").exists() and (chapter9 / "testdata" / "convert_spans_to_whowhen.py").exists():
            return chapter9
    return Path.cwd()


CHAPTER_ROOT = _resolve_chapter_root()

FA_DIR = CHAPTER_ROOT / "Automated_FA"
TESTDATA_DIR = CHAPTER_ROOT / "testdata"
CONVERTED_DIR = TESTDATA_DIR / "converted"
OUTPUT_DIR = FA_DIR / "outputs"

for p in (FA_DIR, TESTDATA_DIR):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

load_dotenv(FA_DIR / ".env")

print("Chapter-9 根目录:", CHAPTER_ROOT.resolve())
print("Automated_FA/.env:", "存在" if (FA_DIR / ".env").exists() else "缺失（请复制 .env.example）")
print("DASHSCOPE_API_KEY:", "已配置" if os.getenv("DASHSCOPE_API_KEY", "").strip() else "未配置")

Chapter-9 根目录: D:\myproject\mira-ai-lab\agent-systems-book\Chapter-9
Automated_FA/.env: 存在
DASHSCOPE_API_KEY: 已配置


## 整体流程

```
spans_*.jsonl          convert_spans_to_whowhen.py          converted/*.json
(你的系统轨迹)    ──►   (Step 1 格式转换)              ──►   (归因输入)
                                                              │
                                                              ▼
                                                    Automated_FA/inference.py
                                                              │
                                                              ▼
                                                    outputs/*.txt
                                                    (Agent / Step / Reason)
                                                              │
                                                              ▼ (可选)
                                                    Automated_FA/evaluate.py
                                                    (与人工标注对比准确率)
```

**history 步骤顺序（转换规则）：**

1. **Planner**：所有 `.planner.*` span，按 `start_time` 排序 → `Planner.{步骤名}`
2. **Agent**：所有 `.agent.invoke` span，按 `execution.order` 排序 → `attributes.task.agent`
3. **汇总**：`.orchestration.aggregate` → `Orchestrator`

**Step 编号** = `history` 数组下标，从 **0** 开始。转换后 JSON 用 `history[i].name` 标识 agent，归因时 `--is_handcrafted` 必须为 **False**。

## 1. 核心函数：三步走 + 可选评估

封装 `convert_spans_to_whowhen`、`inference.py`、`evaluate.py`，与 README 第五节「手动验证流程」一一对应。

In [77]:
import re

from convert_spans_to_whowhen import convert_jsonl_to_whowwhen
from evaluate import evaluate_accuracy, read_predictions

METHOD_CHOICES = ("all_at_once", "step_by_step", "binary_search")
DEFAULT_METHOD = "step_by_step"
DEFAULT_MODEL = "qwen-plus"
IS_HANDCRAFTED = False  # 转换后的 JSON 必须 False，否则会读 history[i].role


def parse_attribution_results(eval_text: str) -> list[dict[str, Any]]:
    """从 outputs/*.txt 解析归因结果。"""
    results: list[dict[str, Any]] = []
    pattern = r"Prediction for ([^\n:]+\.json)\s*[：:](.*?)(?=Prediction for|\Z)"
    for match in re.finditer(pattern, eval_text, re.DOTALL):
        fname = match.group(1).strip()
        block = match.group(2)
        agent_m = re.search(r"(?:Agent Name|错误智能体)[：:]\s*([\w.]+)", block, re.I)
        step_m = re.search(r"(?:Step Number|步骤编号|步骤)[：:]\s*(\d+)", block, re.I)
        reason_m = re.search(
            r"(?:错误原因|Reason provided by LLM|Reason for Mistake)[：:]\s*(.+)",
            block,
            re.I | re.DOTALL,
        )
        results.append({
            "file": fname,
            "agent": agent_m.group(1) if agent_m else "未知",
            "step": step_m.group(1) if step_m else "未知",
            "reason": reason_m.group(1).strip() if reason_m else "（未解析到原因）",
        })
    return results


def print_attribution_summary_zh(
    eval_file: Path,
    converted_dir: Path | None = None,
) -> list[dict[str, Any]]:
    """以中文格式展示归因结果。"""
    if not eval_file.exists():
        print(f"尚未生成归因输出：{eval_file}")
        print("请先运行 Step 2，或执行 inference.py。")
        return []

    eval_text = eval_file.read_text(encoding="utf-8")
    results = parse_attribution_results(eval_text)
    if not results:
        print("未在输出文件中找到归因结果（需包含「Prediction for xxx.json」段落）。")
        return []

    print(f"归因结果文件：{eval_file}\n")
    for i, item in enumerate(results, start=1):
        question = ""
        if converted_dir:
            json_path = converted_dir / item["file"]
            if json_path.exists():
                question = json.loads(json_path.read_text(encoding="utf-8")).get("question", "")

        print(f"{'─' * 60}")
        print(f"【轨迹 {i}】{item['file']}")
        if question:
            q_preview = question.replace("\n", " ")[:80]
            print(f"  用户问题：{q_preview}{'…' if len(question) > 80 else ''}")
        print(f"  错误智能体：{item['agent']}")
        print(f"  错误步骤：{item['step']}（history 下标，从 0 开始）")
        print(f"  错误原因：{item['reason']}")
    print(f"{'─' * 60}")
    print(f"共 {len(results)} 条轨迹完成归因。")
    return results


def _print_conversion_summary(out_path: Path) -> None:
    with out_path.open("r", encoding="utf-8") as f:
        record = json.load(f)
    print(f"Wrote {out_path}")
    print(f"execution_order: {record.get('execution_order', [])}")
    print(f"history steps: {len(record['history'])}")
    for idx, step in enumerate(record["history"]):
        print(f"  Step {idx}: {step['name']}")


def _run_inference(method: str, model: str, directory_path: Path) -> Path:
    """调用 inference.py 子进程，保证输出格式与命令行一致。"""
    output_path = OUTPUT_DIR / f"{method}_{model}_alg_generated.txt"
    cmd = [
        sys.executable,
        str(FA_DIR / "inference.py"),
        "--method", method,
        "--model", model,
        "--is_handcrafted", "False",
        "--directory_path", str(directory_path),
    ]
    print("运行:", " ".join(cmd))
    subprocess.run(cmd, cwd=str(FA_DIR), check=True)
    if not output_path.exists():
        raise FileNotFoundError(f"归因未生成输出: {output_path}")
    return output_path


def print_evaluation_comparison_zh(
    predictions: dict[str, Any],
    converted_dir: Path,
    total_files: int,
) -> None:
    """逐条对比模型预测与人工标注，解释准确率来源。"""
    print("\n【预测 vs 标注 对比】")
    labeled_count = 0
    correct_agent = 0
    correct_step = 0

    for fname, pred in predictions.items():
        json_path = converted_dir / fname
        actual_agent, actual_step = None, None
        if json_path.exists():
            data = json.loads(json_path.read_text(encoding="utf-8"))
            actual_agent = data.get("mistake_agent")
            actual_step = data.get("mistake_step")

        pred_agent = pred["predicted_agent"]
        pred_step = pred["predicted_step"]

        if actual_agent is None or actual_step is None:
            print(f"  {fname}")
            print(f"    预测：{pred_agent} @ 步骤 {pred_step}")
            print(f"    标注：无（未写入 mistake_agent / mistake_step，不参与计分）")
            continue

        labeled_count += 1
        agent_ok = str(actual_agent) in pred_agent or pred_agent in str(actual_agent)
        step_ok = str(actual_step) == str(pred_step)
        if agent_ok:
            correct_agent += 1
        if step_ok:
            correct_step += 1

        mark = "✓" if (agent_ok and step_ok) else "✗"
        print(f"  {mark} {fname}")
        print(f"    预测：{pred_agent} @ 步骤 {pred_step}")
        print(f"    标注：{actual_agent} @ 步骤 {actual_step}")
        if not agent_ok:
            print(f"    → Agent 不匹配")
        if not step_ok:
            print(f"    → 步骤不匹配")

    print(f"\n【计分说明】")
    print(f"  evaluate.py 分母 = converted/ 下全部 JSON 数（当前 {total_files} 条）")
    print(f"  有标注可对比：{labeled_count} 条；Agent 命中 {correct_agent} 条；Step 命中 {correct_step} 条")
    if labeled_count and labeled_count < total_files:
        print(f"  若仅按有标注文件计分：Agent {correct_agent}/{labeled_count}，Step {correct_step}/{labeled_count}")


def run_failure_attribution_eval(
    jsonl_paths: list[Path | str],
    method: str = DEFAULT_METHOD,
    model: str = DEFAULT_MODEL,
    labels: dict[str, dict[str, str]] | None = None,
    run_inference: bool = False,
    run_evaluation: bool = False,
    eval_file: Path | str | None = None,
    skip_convert: bool = False,
) -> dict[str, Any]:
    """AutoFA 流水线：Step 1 转换 → Step 2 归因 → Step 4 评估。

    - Step 1 仅转换：run_evaluation=False（默认）
    - Step 4 评估：run_evaluation=True，需已有 outputs/*.txt 或 run_inference=True
    """
    if method not in METHOD_CHOICES:
        raise ValueError(f"method 必须是 {METHOD_CHOICES}")

    labels = labels or {}
    jsonl_paths = [Path(p) for p in jsonl_paths]
    CONVERTED_DIR.mkdir(parents=True, exist_ok=True)
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    ## Step 1: 将 OpenTelemetry span 轨迹转为 Who&When JSON（可附带 mistake_agent/step 标注）
    converted_files: list[Path] = []
    if not skip_convert:
        for jsonl_path in jsonl_paths:
            if not jsonl_path.exists():
                raise FileNotFoundError(f"轨迹文件不存在: {jsonl_path}")
            label = labels.get(jsonl_path.stem, {})
            out_path = convert_jsonl_to_whowwhen(
                jsonl_path=jsonl_path,
                output_dir=CONVERTED_DIR,
                mistake_agent=label.get("mistake_agent"),
                mistake_step=label.get("mistake_step"),
                mistake_reason=label.get("mistake_reason"),
            )
            converted_files.append(out_path)
            _print_conversion_summary(out_path)
            print()
    else:
        converted_files = sorted(CONVERTED_DIR.glob("*.json"))

    eval_path: Path | None = None
    predictions: dict[str, Any] = {}
    labeled_files = 0
    agent_accuracy, step_accuracy = 0.0, 0.0

    need_eval_output = run_inference or run_evaluation

    ## Step 2: 对 converted/ 下全部 JSON 执行失败归因（需 DASHSCOPE_API_KEY）
    if run_inference:
        if not os.getenv("DASHSCOPE_API_KEY", "").strip():
            raise ValueError("Step 2 需要先在 Automated_FA/.env 配置 DASHSCOPE_API_KEY")
        eval_path = _run_inference(method, model, CONVERTED_DIR)
    elif need_eval_output:
        eval_path = Path(eval_file) if eval_file else OUTPUT_DIR / f"{method}_{model}_alg_generated.txt"
        if not eval_path.exists():
            raise FileNotFoundError(
                f"未找到归因输出: {eval_path}\n"
                "请先运行 Step 2（run_inference=True），或取消注释 Step 2 单元格。"
            )

    ## Step 3 / Step 4: 解析预测；若 JSON 含 mistake_agent/step 则计算准确率
    if run_evaluation and eval_path is not None:
        predictions = read_predictions(str(eval_path))
        total_files = len(list(CONVERTED_DIR.glob("*.json")))
        labeled_files = sum(
            1 for p in CONVERTED_DIR.glob("*.json")
            if json.loads(p.read_text(encoding="utf-8")).get("mistake_agent")
        )

        if labeled_files == 0:
            print("\n提示: converted/*.json 中无 mistake_agent 标注，跳过 Step 4 准确率计算。")
            print("可在 labels 中填写标注后重新 run_failure_attribution_eval。")
        else:
            agent_accuracy, step_accuracy = evaluate_accuracy(
                predictions, str(CONVERTED_DIR), total_files
            )

    return {
        "converted_files": [str(p) for p in converted_files],
        "eval_file": str(eval_path) if eval_path else None,
        "predictions": predictions,
        "labeled_files": labeled_files,
        "agent_accuracy": agent_accuracy,
        "step_accuracy": step_accuracy,
    }

## 2. 样例轨迹与人工标注

仓库内两条测试轨迹（详见 README 第六节）：

| 文件 | 用户问题 | Agent 数量 | 主要问题（人工判断） |
|------|----------|-----------|---------------------|
| `spans_20260615_170343.jsonl` | 大同未来2周的天气如何 | T1（WeatherAgent） | 只返回 14 天预报，缺少完整 2 周 |
| `spans_20260615_180713.jsonl` | 上海/苏州/杭州多城市旅行规划 | T1–T4 | WeatherAgent 查了本周天气而非「下周」 |

`labels` 仅在 **Step 4 评估** 时需要；Step 编号 = `history` 下标，从 0 开始。

In [78]:
SAMPLE_JSONL = [
    TESTDATA_DIR / "spans_20260615_170343.jsonl",
    TESTDATA_DIR / "spans_20260615_180713.jsonl",
]

# 人工标注（README Step 4 示例：仅轨迹 2 有明确标注）
LABELS = {
    "spans_20260615_180713": {
        "mistake_agent": "WeatherAgent",
        "mistake_step": "4",
        "mistake_reason": "查询了本周天气而非用户要求的「下周」",
    },
}

for p in SAMPLE_JSONL:
    print(p.name, "→", "存在" if p.exists() else "缺失")

spans_20260615_170343.jsonl → 存在
spans_20260615_180713.jsonl → 存在


## 3. Step 1 — 轨迹转换

命令行等价：

```powershell
cd Chapter-9
python testdata/convert_spans_to_whowhen.py `
  testdata/spans_20260615_170343.jsonl `
  testdata/spans_20260615_180713.jsonl `
  -o testdata/converted
```

**预期：** 输出 2 个 JSON（文件名 = 输入 stem），不能互相覆盖。

In [79]:
step1 = run_failure_attribution_eval(
    jsonl_paths=SAMPLE_JSONL,
    labels=LABELS,
    run_inference=False,
    run_evaluation=False,  # Step 1 只做转换，不要求 outputs/*.txt
    skip_convert=False,
)
print("已转换:", step1["converted_files"])

Wrote D:\myproject\mira-ai-lab\agent-systems-book\Chapter-9\testdata\converted\spans_20260615_170343.json
execution_order: ['T1']
history steps: 6
  Step 0: Planner.pre_survey
  Step 1: Planner.decomposition
  Step 2: Planner.dependency
  Step 3: Planner.routing
  Step 4: WeatherAgent
  Step 5: Orchestrator

Wrote D:\myproject\mira-ai-lab\agent-systems-book\Chapter-9\testdata\converted\spans_20260615_180713.json
execution_order: ['T1', 'T2', 'T3', 'T4']
history steps: 9
  Step 0: Planner.pre_survey
  Step 1: Planner.decomposition
  Step 2: Planner.dependency
  Step 3: Planner.routing
  Step 4: WeatherAgent
  Step 5: HotelAgent
  Step 6: RestaurantAgent
  Step 7: ItineraryAgent
  Step 8: Orchestrator

已转换: ['D:\\myproject\\mira-ai-lab\\agent-systems-book\\Chapter-9\\testdata\\converted\\spans_20260615_170343.json', 'D:\\myproject\\mira-ai-lab\\agent-systems-book\\Chapter-9\\testdata\\converted\\spans_20260615_180713.json']


## 4. Step 2 — 失败归因（需 API Key，约 30–60 秒）

命令行等价：

```powershell
cd Chapter-9/Automated_FA
python inference.py `
  --method step_by_step `
  --model qwen-plus `
  --is_handcrafted False `
  --directory_path ../testdata/converted
```

输出文件：`outputs/step_by_step_qwen-plus_alg_generated.txt`

三种归因方法：

| 方法 | 说明 |
|------|------|
| `all_at_once` | 一次性看完整 history，最快 |
| `step_by_step` | 逐步扫描，遇第一个「有错」步骤即停止（**推荐**） |
| `binary_search` | 二分定位 step，history 很长时更省 token |

In [80]:
# 取消注释以运行 Step 2（会调用通义千问 API，并覆盖 outputs/*.txt）
step2 = run_failure_attribution_eval(
    jsonl_paths=SAMPLE_JSONL,
    method="step_by_step",
    model="qwen-plus",
    labels=LABELS,
    run_inference=True,
    skip_convert=True,
)
print("归因输出:", step2["eval_file"])

运行: C:\Users\zhanghong26\.conda\envs\agent-systems-book\python.exe D:\myproject\mira-ai-lab\agent-systems-book\Chapter-9\Automated_FA\inference.py --method step_by_step --model qwen-plus --is_handcrafted False --directory_path D:\myproject\mira-ai-lab\agent-systems-book\Chapter-9\testdata\converted
归因输出: D:\myproject\mira-ai-lab\agent-systems-book\Chapter-9\Automated_FA\outputs\step_by_step_qwen-plus_alg_generated.txt


## 5. Step 3 — 查看归因结果

README 实测预期（`step_by_step` + `qwen-plus`）：

| 轨迹文件 | 预测 Agent | 预测 Step | 说明 |
|----------|-----------|-----------|------|
| `spans_20260615_170343.json` | `Planner.dependency` | 2 | LLM 在 Step 2 判定有错并停止 |
| `spans_20260615_180713.json` | `WeatherAgent` | 4 | 与人工分析一致：天气日期范围错误 |

> **注意（FAQ Q2）：** `step_by_step` 遇到**第一个** LLM 判为「有错」的步骤就停止。轨迹 1 在 Step 2 就停了，不会继续扫到 Step 4 的 WeatherAgent。复杂轨迹可改用 `binary_search` 或 `all_at_once`。

In [81]:
EVAL_FILE = OUTPUT_DIR / "step_by_step_qwen-plus_alg_generated.txt"

attribution_results = print_attribution_summary_zh(EVAL_FILE, CONVERTED_DIR)

归因结果文件：D:\myproject\mira-ai-lab\agent-systems-book\Chapter-9\Automated_FA\outputs\step_by_step_qwen-plus_alg_generated.txt

────────────────────────────────────────────────────────────
【轨迹 1】spans_20260615_170343.json
  用户问题：大同未来2周的天气如何
  错误智能体：Orchestrator
  错误步骤：5（history 下标，从 0 开始）
  错误原因：Orchestrator 在最终响应中错误地将用户“未来2周”的时间范围等同于「2026-06-15 至 2026-07-01（17天）」，但根据系统时间锚点（今天：2026-06-15），「未来2周」严格指从今日起向后推14天，即 **2026-06-15 至 2026-06-28（含首尾共14天）**，而非到7月1日。2026-06-29 至 2026-07-01 属于第15–17天，已超出“未来2周”范畴，属于 Planner/WeatherAgent 在步骤2–4中错误扩展了时间窗口（应设 days=14、end_date=2026-06-28），而 Orchestrator 未纠正该偏差，反而在响应中主动强调“用户请求的是…至2026-07-01（17天）”，将错误前提当作事实陈述，误导用户并掩盖真正可交付范围——这直接导致对用户意图的严重误读，构成实质性错误。


--- 正在分析：spans_20260615_180713.json ---
正在评估第 0 步（Planner.pre_survey）...
模型判定：1. 否  
2. 原因：Planner.pre_survey 步骤仅输出结构化分析框架（如待查事实、待推导项等），未涉及具体时间推算、地点判断或工具调用；其返回的 JSON 结构完整，字段合理（"given_facts"可提取“下周”“上海、苏州、杭州”“安静酒店”“≤800元/晚”等关键约束；"facts_to_lookup"应涵盖三地交通、天气、酒店等，符合用户需求；所有时间理解均以系统锚点「2026-06-15」为基准，“下周”即2026-06-16至202

## 6. Step 4（可选）— 评估准确率

命令行等价：

```powershell
cd Chapter-9/Automated_FA
python evaluate.py `
  --data_path ../testdata/converted `
  --eval_file outputs/step_by_step_qwen-plus_alg_generated.txt
```

准确率分母 = `converted/` 下全部 JSON 数量；若仅 1 条有标注，最高 50%（2 条轨迹时）。

In [82]:
try:
    result = run_failure_attribution_eval(
        jsonl_paths=SAMPLE_JSONL,
        method="step_by_step",
        model="qwen-plus",
        labels=LABELS,
        run_inference=False,
        run_evaluation=True,
        skip_convert=True,
    )
except FileNotFoundError as exc:
    print(exc)
    result = {"eval_file": None, "labeled_files": 0, "predictions": {}, "agent_accuracy": 0.0, "step_accuracy": 0.0}

print("\n=== Step 4 评估结果 ===")
print(f"归因输出: {result['eval_file']}")
print(f"有标注的文件数: {result['labeled_files']}")
if result["labeled_files"]:
    print(f"Agent 准确率: {result['agent_accuracy']:.2f}%")
    print(f"Step 准确率:  {result['step_accuracy']:.2f}%")
if result["predictions"]:
    total_files = len(list(CONVERTED_DIR.glob("*.json")))
    print_evaluation_comparison_zh(result["predictions"], CONVERTED_DIR, total_files)
else:
    print("\n（无有效预测，请先完成 Step 2 生成 outputs/*.txt）")

--- Predictions Read from D:\myproject\mira-ai-lab\agent-systems-book\Chapter-9\Automated_FA\outputs\step_by_step_qwen-plus_alg_generated.txt ---
Successfully parsed predictions for 2 files.

--- Starting Evaluation ---
Total reference JSON files found in D:\myproject\mira-ai-lab\agent-systems-book\Chapter-9\testdata\converted: 2
Predictions available for 2 files.
Skipping evaluation for spans_20260615_170343.json due to issues reading actual data.

--- Evaluation Summary ---
Total reference files in data_path: 2
Predictions parsed from eval file:  2
Files evaluated (prediction found & actual data read): 2
Correct Agent Predictions: 1
Correct Step Predictions:  1

=== Step 4 评估结果 ===
归因输出: D:\myproject\mira-ai-lab\agent-systems-book\Chapter-9\Automated_FA\outputs\step_by_step_qwen-plus_alg_generated.txt
有标注的文件数: 1
Agent 准确率: 50.00%
Step 准确率:  50.00%

【预测 vs 标注 对比】
  spans_20260615_170343.json
    预测：Orchestrator @ 步骤 5
    标注：无（未写入 mistake_agent / mistake_step，不参与计分）
  ✓ spans_20260615

## 7. 支持的模型与相关文件

| 类型 | `--model` 取值 | 是否需要 Key |
|------|----------------|-------------|
| 通义千问 API | `qwen-plus`, `qwen-max`, `qwen-turbo`, `qwen-long` | 需要 `DASHSCOPE_API_KEY` |
| Azure GPT | `gpt-4o`, `gpt4`, `gpt4o-mini` | 需要 `AZURE_OPENAI_*` |
| 本地 HuggingFace | `qwen-7b`, `qwen-72b`, `llama-8b`, `llama-70b` | 不需要 Key，需要 GPU |

| 文件 | 作用 |
|------|------|
| `testdata/spans_*.jsonl` | 样例轨迹 |
| `testdata/convert_spans_to_whowhen.py` | span JSONL → Who&When JSON |
| `testdata/converted/*.json` | 转换输出（归因输入） |
| `Automated_FA/inference.py` | 归因入口 |
| `Automated_FA/evaluate.py` | 准确率评估 |
| `Automated_FA/Lib/utils.py` | 三种 AutoFA 方法实现 |
| `Automated_FA/outputs/step_by_step_qwen-plus_alg_generated.txt` | 归因输出（实测生成） |
| `testdata/README.md` | 完整使用指南 |